# Deep Q-Learning

We're going to use Deep Q-Learning in order to learn a [cartpole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) agent.  You'll notice the cartpole state space is continuous... Tabular Q-Learning won't work!

## Step 0: Environment setup

Activate your environment from last week, and then run: `pip install gymnasium[classic_control] torch torchvision`. Download [this file](sampled_states.npy) to your code directory and rename it `sampled_states.npy`.

[Here's your Gem](https://gemini.google.com/gem/144socMiRUVdi50ubzaJgFhG7e6cEc6K7?usp=sharing).

We'll now import a bunch of stuff and define some useful variables.

In [ ]:
import gymnasium as gym
import numpy as np
import numpy.random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque, namedtuple
import random
import plotly.graph_objects as go
from tqdm import tqdm

In [ ]:
env = gym.make('CartPole-v1')
STATE_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n
GAMMA = .99

print(f'State space is continuous in {STATE_DIM} dimensions, and there are {N_ACTIONS} actions.')

obs, info = env.reset()
print(f'For example, heres an observation: {obs}.')

## Step 1: Hyperparameters and network definition

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; color: #856404;">
  <strong>🟡 AI Policy: YELLOW</strong> <br>
  Generative AI is allowed, with limitations.
</div>

We have to decide a few things.
- What should $\epsilon$ be for our $\epsilon$-greedy exploration policy?
- How large should our replay be?
- How many datapoints should we pull from our replay to train on at a time (batch size)?
- What should our neural network look like? A good first step here is to make sure you understand what the dimensions of the input and output layers need to be - those aren't up to us, they are prescribed by the problem.

Choose some values, design your network.

Below we've also created a Replay - note that it's essentially a `deque` of limited size (remember linked lists? We're adding onto one end, and removing from the other...).  Make sure you understand that code!  In that replay, we are storing Transitions, each of which consists of a state, an action, a reward, and a next_state.  If the transition represents a failure (pole fell over or cart went off screen), the next_state will be `None`, 

In [ ]:
EPSILON =
REPLAY_LENGTH =
BATCH_SIZE =
LEARNING_RATE =

In [ ]:
# Define a network and create an instance of it

In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'reward', 'next_state'))

class ReplayMemory:

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, s, a, r, sp):
        """Save a transition"""
        s = torch.tensor(s, dtype=torch.float32)
        a = torch.tensor([a], dtype=torch.int64)
        r = torch.tensor([r], dtype=torch.float32)
        if sp is not None:
            sp = torch.tensor(sp, dtype=torch.float32)
        self.memory.append(Transition(s,a,r,sp))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

memory=ReplayMemory(REPLAY_LENGTH)

## Policies!

The below functions define a greedy policy, an $\epsilon$-greedy policy, and a random policy.

In [ ]:
def greedy_policy(network, states):
    '''
    Returns a tuple
    index 0 contains the Q-value of the best action for all states
    index 1 contains the index of the best action for all states
    '''
    with torch.no_grad():
        qs = network(states) # Get the q-values
        if qs.dim() == 1:   # If it's just a single state
            return torch.max(qs, dim=0) # Return the tuple of max info for that state
        return torch.max(qs, dim=1) # Return the tuple of max information for all states

def epsilon_greedy(network, state):
    '''
    Returns an action selected via epsilon-greedy
    '''
    if numpy.random.random() < EPSILON:
        return numpy.random.randint(N_ACTIONS)
    else:
        return greedy_policy(network, torch.tensor(state).to('cuda'))[1].item()

def random_policy():
    '''
    Chooses a random action.
    '''
    return numpy.random.randint(N_ACTIONS)

## Step 2: Quantifying random performance

100 times, reset the environment and run it until truncation or termination, using a random policy.  Print out the average number of steps a random policy keeps the pole upright.

## Step 3: Training

<div style="background-color: #d4edda; border-left: 6px solid #28a745; padding: 15px; color: #155724;">
  <strong>🟢 AI Policy: GREEN</strong> <br>
  Generative AI is allowed/encouraged for this section.
</div>

We're going to reproduce the graphs on the right-hand side of Figure 2 in the paper in order to judge the smoothness of our training.  Create a function called `avg_qs`, which accepts as arguments your network and a tensor representing a group of states, which you should load from `sampled_states.npy`.  It should then do the following:

- in a `with torch.no_grad()` block, push the states through the network, producing some approximate Q-values
- calculate the maximum Q-value for each state
- average those maximum Q-values over all the states, resulting in a single scalar
- return that scalar (which should just be a number, not a tensor or numpy array)

In [ ]:
sampled_states = torch.tensor(np.load('sampled_states.npy'), dtype=torch.float32).to('cuda')

def avg_qs(network, states):


## Training your network

Create an optimizer and choose a loss criterion. 

Understand, then complete, the `train_model()` function. This function implements the gradient descent step described in Algorithm 1, specifically calculating the targets $y_j$.

**Handling Terminal States**

Algorithm 1 defines the target $y_j$ differently depending on whether the transition leads to a terminal state or a non-terminal state. A terminal state occurs when the episode ends, meaning there are no future rewards to collect. 

* For terminal next states, the target $y_j$ is strictly the immediate reward $r_j$.
* For non-terminal next states, the target $y_j$ is the immediate reward $r_j$ plus the discounted maximum Q-value of the next state $r_j + \gamma \max_{a'} Q(\phi_{j+1}, a'; \theta)$.

**Code Implementation Details**

In the provided starter code, terminal states are represented by a `None` value for the `next_state`. The initial steps in `train_model()` separate the transitions to make this calculation efficient using tensors:

* `states`, `actions`, and `rewards` are tensors containing the data for the entire batch.
* `non_terminal_mask` is a boolean tensor indicating which transitions in the batch have a valid next state (True for non-terminal, False for terminal).
* `non_terminal_next_states` is a tensor containing only the valid, non-terminal next states. 

You must use `non_terminal_mask` and `non_terminal_next_states` to compute the maximum future Q-values solely for the non-terminal transitions. Initialize a tensor for future Q-values with zeros, then use the boolean mask to insert the computed maximum Q-values for the non-terminal states. Finally, add these discounted future Q-values to your immediate `rewards` to compute the target $y_j$ for the entire batch simultaneously.

In [ ]:
optimizer = # Initialize your optimizer here
criterion = # Initialize your loss function here

def train_model():
    if len(memory) < BATCH_SIZE:
        return
    
    transitions = memory.sample(BATCH_SIZE)

    # 1. Isolate components of the transition batch
    # states shape: (BATCH_SIZE, STATE_DIM)
    # actions shape: (BATCH_SIZE, 1)
    # rewards shape: (BATCH_SIZE, 1)
    states = torch.cat([t.state.unsqueeze(0) for t in transitions], dim=0).to('cuda')
    actions = torch.cat([t.action.unsqueeze(0) for t in transitions], dim=0).to('cuda')
    rewards = torch.cat([t.reward.unsqueeze(0) for t in transitions], dim=0).to('cuda')

    # 2. Handle Terminal vs. Non-Terminal Next States
    # A state is terminal if the environment returned None for the next_state.
    # non_terminal_mask is a boolean tensor of shape (BATCH_SIZE,) where True indicates a non-terminal state.
    non_terminal_mask = torch.tensor(
        tuple(map(lambda t: t.next_state is not None, transitions)), 
        device='cuda', 
        dtype=torch.bool
    )

    # Extract the actual next states exclusively for the non-terminal transitions.
    # non_terminal_next_states shape: (NUMBER_OF_NON_TERMINAL_STATES, STATE_DIM)
    non_terminal_next_states = torch.cat(
        [t.next_state.unsqueeze(0) for t in transitions if t.next_state is not None], 
        dim=0
    ).to('cuda')

    # 3. Calculate Targets (y_j) according to Algorithm 1
    # - Initialize a tensor for next state Q-values with zeros (shape: BATCH_SIZE).
    # - Pass non_terminal_next_states through your target network to find the max future Q-values.
    # - Place those max future Q-values into the zero tensor at the indices specified by non_terminal_mask.
    # - Calculate expected Q values: y_j = rewards + (GAMMA * next state Q-values).
    #   (For terminal states, the next state Q-value remains 0, leaving strictly the reward).

    # 4. Compute Loss and Optimize
    # - Pass the current states through your policy network and gather the Q-values for the taken actions.
    # - Calculate the loss between these predicted Q-values and your y_j targets.
    # - Perform backpropagation and step the optimizer.

## Create your samples, and call the training function

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; color: #856404;">
  <strong>🟡 AI Policy: YELLOW</strong> <br>
  Generative AI is allowed, with limitations.
</div>

Implement the rest of Algorithm 1, calling your `train_model()` function where appropriate.

If the transition is terminal, the next_state should be `None`.

## Evaluating the smoothness of your training

In the above training loop, keep track of the average maximum Q values for the `sampled_states` you loaded above.  Make a plot displaying the average Q values of the sampled states over time.

In [ ]:
fig = go.Figure(data = go.Scatter(x=list(range(len(qs))), y=qs, mode='lines'))
fig.show()

## Evaluating your model's performance

1000 times, use a greedy policy based off your model to run until termination or truncation.  Keep track of the number of steps that pass on each run before it stops (each trial will run a maximum of 500 steps before truncation - of course, it may terminate sooner if the pole falls or the cart goes off the screen).  Print out that average.